In [ ]:
#0-1 EDAセッティング　基本設定

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
%config Inlinebackend.figure_format='retina'
sns.set(style='whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#0-2 EDAセッティング　メモリ削減（ダウンキャスト）

total_mem = 0
total_mem_r = 0

def reduce(df):
    global total_mem,total_mem_r

    """ 数値データの型を最適化してメモリを節約する関数 """
    start_mem=df.memory_usage().sum()/1024**2

    for col in df.columns:
      col_type=df[col].dtype
      if col_type != object:
          cmin=df[col].min()
          cmax=df[col].max()

          if str(col_type)[:3]=='int':
            if   cmin > np.iinfo(np.int8).min and cmax < np.iinfo(np.int8).max:
              df[col]=df[col].astype(np.int8)
            elif cmin > np.iinfo(np.int16).min and cmax < np.iinfo(np.int16).max:
              df[col]=df[col].astype(np.int16)
            elif cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
              df[col]=df[col].astype(np.int32)
            else:
              df[col]=df[col].astype(np.int64)
          else:
            if   cmin > np.finfo(np.float16).min and cmax < np.finfo(np.float16).max:
              df[col]=df[col].astype(np.float16)
            elif cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
              df[col]=df[col].astype(np.float32)
            else:
              df[col]=df[col].astype(np.float64)

    end_mem=df.memory_usage().sum()/1024**2
    print(f'メモリ使用量: {start_mem:.2f}MB -> {end_mem:.2f}MB | ',
          f'削減: {(start_mem - end_mem) / start_mem:.2f}MB {100*(start_mem - end_mem) / start_mem:.1f}%')
    total_mem +=start_mem
    total_mem_r +=end_mem
    return df

In [ ]:
#0-3 EDAセッティング　csv読込
id_train=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/train_identity.csv')
id_train=reduce(id_train)
id_test=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/test_identity.csv')
id_test=reduce(id_test)

tra_train=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/train_transaction.csv')
tra_train=reduce(tra_train)
tra_test=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/test_transaction.csv')
tra_test=reduce(tra_test)

sample=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/sample_submission.csv')
sample=reduce(sample)

total_reduce = total_mem - total_mem_r
total_ratio = 100 * (total_reduce/total_mem)
print('-'*20)
print(f'合計メモリ使用量：{total_mem:.2f}MB -> {total_mem_r:.2f}MB | ',
      f'合計削減: {(total_reduce):.2f}MB {(total_ratio):.2f}%')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Kaggle/IEEE-CIS Fraud Detection/train_identity.csv'

In [ ]:
#0-4 EDAセッティング　df確認1(ID)
display(pd.concat([id_train.head(4),id_train.tail(2)]))

In [ ]:
#0-4 EDAセッティング　df確認2(トランザクション)
display(pd.concat([tra_train.head(10),tra_train.tail()]))

In [ ]:
#0-4 EDAセッティング　df確認3(カラム説明　Kaggleより引用)

'''
【アイデンティティテーブル】
このテーブル内の変数は、取引に関連付けられた識別情報（ネットワーク接続情報
（IP、ISP、プロキシなど）とデジタル署名（UA/ブラウザ/OS/バージョンなど））
です。
これらは、Vestaの不正防止システムとデジタルセキュリティパートナーによって収
集されます。
（フィールド名はマスクされており、プライバシー保護と契約締結のため、ペアワイ
ズ辞書は提供されません。）

カテゴリ別特徴: DeviceType DeviceInfo id_12 - id_38

【取引表】
TransactionDT: 指定された参照日時からのタイムデルタ（実際のタイムスタンプで
　　　　　　　 はありません）
TransactionAMT: 取引支払金額（米ドル）
ProductCD: 製品コード、各トランザクションの製品
card1 - card6: カードの種類、カードのカテゴリ、発行銀行、国などの支払いカー
               ド情報。
addr: 住所
dist: 距離
P_ および (R__) emaildomain: 購入者と受信者のメールドメイン
C1-C14: 支払いカードに関連付けられているアドレスがいくつ見つかったかなどのカ
　　　　ウント。実際の意味はマスクされています。
D1-D15: 前回の取引間の日数などの時間差分。
M1-M9: カード名や住所などが一致します。
Vxxx: Vesta は、ランキング、カウント、その他のエンティティ関係を含む豊富な機
　　　能を設計しました。
カテゴリカル特徴: ProductCD card1 - card6 addr1, addr2 P_emaildomain
                  R_emaildomain M1 - M9

'''

In [ ]:
#0-4 EDAセッティング　df確認4(提出データ)
sample

In [ ]:
#0-5 EDAセッティング　欠損値確認(train)

id_and_tra=[('Identity',id_train),('Transaction',tra_train)]

def missing(id_and_tra):

  for name,df in id_and_tra:
    null_counts=df.isnull().sum()
    null_columns=null_counts[null_counts > 0]
    mis_ratio=len(null_columns)/len(df.columns)*100

    if null_counts.empty:
        print('欠損値なし')
    else:
        print(f'【{name}: 欠損あり(全{len(df.columns)}カラム中) {len(null_columns)}件】 ({mis_ratio:.2f}%)')
        tmp_df=null_columns.to_frame(name='ncounts')
        tmp_df['ratio']=(tmp_df['ncounts'] / len(df)) * 100
        df_ratio_desc=tmp_df.sort_values('ratio',ascending=False)

        for ncol,row in df_ratio_desc.iterrows():
            count = row['ncounts']
            ratio = row['ratio']
            print(f'カラム名: {ncol:<15} | 欠損数: {int(count):>8} / 全レコード数: {len(df)} | 割合: {ratio:>6.2f}%')
        print('*'*100)

  return df_ratio_desc

mis_train=missing(id_and_tra)

９割超のカラムが欠損値あり
マージ後のデータで再度要検討

In [ ]:
#0-5 EDAセッティング　無欠損値確認(train)

def not_missing(id_and_tra):

  for name, df in id_and_tra:
      null_counts = df.isnull().sum()
      clean_columns = null_counts[null_counts == 0].index.tolist()

      print(f"--- 【{name}】 無欠損カラム一覧 (計 {len(clean_columns)}件) ---")
      if len(clean_columns) > 0:
          for i in range(0, len(clean_columns), 5):
              print(", ".join(clean_columns[i:i+5]))
      else:
        print("無欠損なし")
      print("\n" + "="*50 + "\n")

  return id_and_tra
not_mis_train=not_missing(id_and_tra)

isFraud欠損なしを確認、また結合キー(TransactionID)も欠損なし

In [ ]:
#0-5 EDAセッティング　欠損値確認(test)

test_mis_nmis=[('Identity',id_test),('Transaction',tra_test)]
not_mis_test=not_missing(test_mis_nmis)
mis_test=missing(test_mis_nmis)

In [ ]:
#0-6 EDAセッティング　結合キー重複確認

for name,df in id_and_tra:
  df=df.groupby('TransactionID').size()
  print(f'【{name}】{df.sum()}件')
  df_max=df.max()

  if df_max > 1:
    print(f'【{name}】重複あり')
    print(f'最大重複回数:{df_max}')
  else:
    print(f'【{name}】重複なし')

In [ ]:
#0-7 EDAセッティング　結合(train & test)

train = pd.merge(tra_train,id_train,on='TransactionID',how='left')
test = pd.merge(tra_test,id_test,on='TransactionID',how='left')

print(f'レコード数(train)：{train.shape[0]}')
print(f'カラム数(train)：{train.shape[1]}')
print(f'レコード数(test)：{test.shape[0]}')
print(f'カラム数(test)：{test.shape[1]}')

train.head()

カラムの差:isFraud

In [ ]:
#0-7 EDAセッティング　再ダウンキャスト及び出力（パーケ形式）
train=reduce(train)
test=reduce(test)

train.to_parquet('train_merged.parquet')
test.to_parquet('test_merged.parquet')

マージ処理により、サイズがアップキャストされる可能性があるため再ダウンキャストを実施。  
今回は初期の処理により削減なし。


In [ ]:
'''
後続処理
①下処理前EDA
②欠損処理の検討
③特徴量変換(ビン分け、外れ値処理、ラベル化の検討)
④下処理後EDA
'''